In [ ]:
# %% Deep learning - Section 20.191
#    CNN project 2: cifar10 autoencoder
#    1) Import the CIFAR10 dataset, and apply transformations
#    2) Make sure the data are 3x32x32
#    3) Build a CNN autoencoder
#    4) Use the following architecture:
#       - input (3x32x32)
#       - enc_conv1 (16x16x16), k=4, p=1, s=2
#       - enc_conv2 (32x8x8), k=4, p=1, s=2
#       - latent (64x4x4), k=4, p=1, s=2
#       - dec_conv1 (32x8x8), k=4, p=1, s=2
#       - dec_conv2 (16x16x16), k=4, p=1, s=2
#       - output (3x32x32)
#    5) You are free to pick other metaparameters (activation functions, lr,
#       optimizer, epochs, etc.)
#    6) Split in train, dev and test, plot the train and dev losses
#    7) Plot original vs. recostructed data (you might have to clip some small
#       negative values when plotting the AE images)

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% Import data

cifar10_data = torchvision.datasets.CIFAR10(root='cifar10',download=True)
print(cifar10_data)


In [ ]:
# %% Inspect dataset

# Shape of the dataset (num of images, [size, size], RGB channels)
print("Data shape :")
print(cifar10_data.data.shape), print()

# Unique categories
print("Categories :")
print(cifar10_data.classes), print()

# .targets is a list of targets converted to integers
print("First few targets (int) :")
print(cifar10_data.targets[:10]), print()

print("Target length :")
print(len(cifar10_data.targets))


In [13]:
# %% Split data

# Define train, dev, test splits (80/10/10)
indices = np.arange(len(cifar10_data.targets))
np.random.shuffle(indices)

train_boundary = int(0.8 * len(indices))
dev_boundary   = int(0.9 * len(indices))

train_idx = indices[:train_boundary]
dev_idx   = indices[train_boundary:dev_boundary]
test_idx  = indices[dev_boundary:]


In [23]:
# %% Transformation to tensor and data augmentation

# Define transforms (do not normalise to maintain [0,1] range given by the
# .ToTensor() method)
train_transform = T.Compose([ T.ToPILImage(),
                              T.RandomRotation((-15,15)),
                              T.RandomAffine(degrees=0,translate=(0.1,0.1),shear=10),
                              T.ToTensor() ])

test_transform = T.Compose([ T.ToPILImage(),
                             T.ToTensor() ])


In [24]:
# %% Custom dataset

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self,dataset,indices,transform=None):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __getitem__(self,idx):
        x,y = self.dataset[self.indices[idx]]
        if self.transform:
            x = self.transform(x)
        return x,y

    def __len__(self):
        return len(self.indices)


In [25]:
# %% Define sets and apply transforms

# Split data, convert to PyTorch datasets, and apply transforms
train_dataset = CustomDataset(cifar10_data, train_idx, train_transform)
dev_dataset   = CustomDataset(cifar10_data, dev_idx,   test_transform)
test_dataset  = CustomDataset(cifar10_data, test_idx,  test_transform)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,drop_last=True)
dev_loader   = DataLoader(dev_dataset,batch_size=len(dev_dataset))
test_loader  = DataLoader(test_dataset,batch_size=len(test_dataset))


In [ ]:
# %% Check DataLoaders

images, labels = next(iter(train_loader))
print("Train batch (img;labels):", images.shape, labels.shape)

images, labels = next(iter(dev_loader))
print("Dev set (img;labels):", images.shape, labels.shape)

images, labels = next(iter(test_loader))
print("Test set (img;labels):",images.shape,labels.shape)


In [26]:
# %% Function to generate the model

# For the decoder we do "transpose" convolution
def gen_model():

    class cifar10_AE(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(3,16,4,stride=2,padding=1),
                                          nn.ReLU(),
                                          nn.Conv2d(16,32,4,stride=2,padding=1),
                                          nn.ReLU(),
                                          nn.Conv2d(32,64,4,stride=2,padding=1) )

            # Decoding layer (use sigmoid at the end to enforce range [0,1], if
            # you normalise, you might consider Tanh() or no function)
            self.decoder = nn.Sequential( nn.ConvTranspose2d(64,32,4,stride=2,padding=1),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(32,16,4,stride=2,padding=1),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(16,3,4,stride=2,padding=1),
                                          nn.Sigmoid() )

        def forward(self,x):
            x     = self.encoder(x)
            x_hat = self.decoder(x)
            return x_hat

    # Create model instance
    CNN = cifar10_AE()

    # Loss function
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

X, _ = next(iter(train_loader))

CNN,loss_fun,optimizer = gen_model()

X_hat = CNN(X)

print("Input shape :", X.shape)
print("Output shape:", X_hat.shape)

# Input should be [0,1], untrained output should be ~[0.5,0.5]
print("Input range :", X.min().item(), X.max().item())
print("Output range:", X_hat.min().item(), X_hat.max().item())


In [30]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 20
    CNN,loss_fun,optimizer = gen_model()

    # Ship model to GPU
    CNN.to(device)

    train_losses = []
    dev_losses   = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_loss = []

        for X,_ in train_loader:

            # Ship data to GPU
            X = X.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,X)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss
            batch_loss.append(loss.item())

        train_losses.append(np.mean(batch_loss))

        # Dev loss
        CNN.eval()

        with torch.no_grad():
            X,_ = next(iter(dev_loader))

            # Ship to GPU
            X = X.to(device)

            yHat = CNN(X)
            loss = loss_fun(yHat,X)

            dev_losses.append(loss.item())

        CNN.train()

    return train_losses,dev_losses,CNN


In [31]:
# %% Run the model

# Takes ~10 mins on GPU
train_losses,dev_losses,CNN = train_model()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(train_losses,'s-',label='Train')
plt.plot(dev_losses,'s-',label='Dev')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Model loss\n(final dev loss=%.5f)'%dev_losses[-1])

plt.savefig('figure7_cnn_project_2.png')
plt.show()
files.download('figure7_cnn_project_2.png')


In [ ]:
# %% Plotting

# Get test data
X,y = next(iter(test_loader))
X = X.to(device)

with torch.no_grad():
    X_hat = CNN(X)

# Pick 10 random images
rand_id = np.random.choice(len(X), size=10, replace=False)

# Colormap to map correlations: red (0) to green (1)
cmap = plt.cm.RdYlGn

# Plot
phi = (1 + np.sqrt(5)) / 2
fig, axs = plt.subplots(2, 10, figsize=(2*phi*7,7))

for i in range(10):
    idx = rand_id[i]

    # Original
    img = X[idx].permute(1,2,0).cpu().numpy()

    label = cifar10_data.classes[y[idx].item()]

    axs[0,i].imshow(img)
    axs[0,i].text(16,0,label,ha='center',fontweight='bold',color='k',backgroundcolor='y')
    axs[0,i].axis('off')

    # Reconstructed (with correlation)
    img_hat = X_hat[idx].permute(1,2,0).cpu().numpy()

    corr  = np.corrcoef(img.flatten(), img_hat.flatten())[0,1]
    color = cmap(corr)

    axs[1,i].imshow(img_hat)
    axs[1,i].text(16,0,f"r={corr:.2f}",ha='center',fontweight='bold',color='k',backgroundcolor=color)
    axs[1,i].axis('off')

fig.text(0.11, 0.72, "Original", va='center', rotation='vertical', fontsize=12,fontweight='bold')
fig.text(0.11, 0.29, "Reconstructed", va='center', rotation='vertical', fontsize=12,fontweight='bold')

plt.savefig('figure8_cnn_project_2.png')
plt.show()
files.download('figure8_cnn_project_2.png')
